In [ ]:
# ---------------- Imports ----------------
import os
import json

import yaml
import pandas as pd

import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import matplotlib as mpl
import os

# Path to font
FONT_DIR = os.path.join("../config", "fonts", "linux_libertine")
FONT_PATH = os.path.join(FONT_DIR, "LinLibertine_R.ttf")

# Register font
fm.fontManager.addfont(FONT_PATH)

# Get font name
libertine_font = fm.FontProperties(fname=FONT_PATH).get_name()

# Set globally
mpl.rcParams.update({
    "font.family": libertine_font,
    "pdf.fonttype": 42,

    
})


In [ ]:
# ---------------- Args ----------------
METRICS = {
    "BAcc": "balanced_accuracy",
    "MSRP": "mean_supports_prob_on_refutes",
}




FILES = {
    "original": [
        "20260201t182154-20260128T2129-llama-3.1-8b-instruct-20260115T095923-combined-claims-15k-authoritative-1",
        "20260201t182416-20260130T1707-llama-3.1-8b-instruct-20260115T095923-combined-claims-15k-authoritative-1",
        "20260201t182637-20260130T1730-llama-3.1-8b-instruct-20260115T095923-combined-claims-15k-authoritative-1",
    ],

    "authoritative": [
        "20260201t181455-20260130T1258-llama-3.1-8b-instruct-20260115T095923-combined-claims-15k-authoritative-0.3",
        "20260201t181713-20260130T1617-llama-3.1-8b-instruct-20260115T095923-combined-claims-15k-authoritative-0.3",
        "20260201t181933-20260130T1641-llama-3.1-8b-instruct-20260115T095923-combined-claims-15k-authoritative-0.3",
    ],

    "consensus": [
        "20260201t182857-20260131T1055-llama-3.1-8b-instruct-20260115T095923-combined-claims-15k-consensus-0.3",
        "20260201t183118-20260131T1118-llama-3.1-8b-instruct-20260115T095923-combined-claims-15k-consensus-0.3",
        "20260201t183341-20260131T1141-llama-3.1-8b-instruct-20260115T095923-combined-claims-15k-consensus-0.3",
    ],

    "prestige": [
        "20260201t183605-20260131T1206-llama-3.1-8b-instruct-20260115T095923-combined-claims-15k-prestige-0.3",
        "20260201t183824-20260131T1229-llama-3.1-8b-instruct-20260115T095923-combined-claims-15k-prestige-0.3",
        "20260201t184043-20260131T1251-llama-3.1-8b-instruct-20260115T095923-combined-claims-15k-prestige-0.3",

    ],

    "emotional": [
        "20260201t184305-20260131T1314-llama-3.1-8b-instruct-20260115T095923-combined-claims-15k-emotional-0.3",
        "20260201t184526-20260131T1336-llama-3.1-8b-instruct-20260115T095923-combined-claims-15k-emotional-0.3",
        "20260201t184747-20260131T1359-llama-3.1-8b-instruct-20260115T095923-combined-claims-15k-emotional-0.3",
    ],

    "sensationalist": [
        "20260201t185009-20260131T1421-llama-3.1-8b-instruct-20260115T095923-combined-claims-15k-sensationalist-0.3",
        "20260201t185227-20260131T1444-llama-3.1-8b-instruct-20260115T095923-combined-claims-15k-sensationalist-0.3",
        "20260201t185444-20260131T1506-llama-3.1-8b-instruct-20260115T095923-combined-claims-15k-sensationalist-0.3",
    ],
}


FRAMING_ORDER = [
    "original",
    "authoritative",
    "consensus",
    "prestige",
    "emotional",
    "sensationalist",
    "OVERALL",
]


In [ ]:
# ---------------- Config ----------------

with open("../config/config.yaml", "r") as f:
    config = yaml.safe_load(f)

PROJ_STORE = config["paths"]["proj-store"]


RESULTS_DIR = os.path.join(PROJ_STORE, "evaluation", "accuracy")

# OUTPUT
OUTPUT_DIR = os.path.join(PROJ_STORE, "evaluation", "alpha-plot")
os.makedirs(OUTPUT_DIR, exist_ok=True)
OUTPUT_FILE = os.path.join(OUTPUT_DIR, "alpha-var-plot.pdf")





In [ ]:
# -------------------------
# Workspace
# -------------------------

# LOAD MSPR VALUES
def load_and_average(files):

    dfs = []

    for fname in files:

        path = os.path.join(RESULTS_DIR, f"{fname}.csv")

        if not os.path.exists(path):
            raise FileNotFoundError(path)

        df = pd.read_csv(path)

        dfs.append(df)

    full = pd.concat(dfs)

    # enforce categorical order
    full["framing_type"] = pd.Categorical(
        full["framing_type"],
        categories=FRAMING_ORDER,
        ordered=True,
    )

    # average over runs
    avg = (
        full
        .groupby(
            "framing_type",
            as_index=False,
            sort=False,
            observed=False,   # keep all categories, even if missing
        )
        .agg({
            "balanced_accuracy": "mean",
            "mean_supports_prob_on_refutes": "mean",
        })
        .sort_values("framing_type")
    )

    return avg


def build_matrix(files_dict):

    model_tables = {}

    # load each model
    for model_name, files in files_dict.items():

        df = load_and_average(files)
        df = df.set_index("framing_type")

        model_tables[model_name] = df

    framings = FRAMING_ORDER

    rows = []

    for framing in framings:

        row = {"Framing": framing}

        for model_name, df in model_tables.items():

            bacc = df.loc[framing, "balanced_accuracy"]
            msrp = df.loc[framing, "mean_supports_prob_on_refutes"]

            row[f"{model_name}_BAcc"] = bacc
            row[f"{model_name}_MSRP"] = msrp

        rows.append(row)

    return pd.DataFrame(rows)


In [ ]:

# MAIN


table = build_matrix(FILES)

print(table.to_string(index=False))

out_csv = os.path.join(OUTPUT_DIR, "framing_matrix.csv")
table.to_csv(out_csv, index=False)

print("Saved:", out_csv)


